# Лабораторна робота № 5
## Географічні дані OpenStreetMap: від тегів до маршруту

**Інтелектуальний аналіз та візуалізація даних** · 4 години · 4 бали

---

### Постановка

Ви — аналітик у туристичному застосунку. Завдання: для заданого міста
**відібрати десять найвартісніших історико-культурних об'єктів** і
**спланувати піший маршрут**, який починається на залізничному вокзалі,
обходить усі десять і повертається на вокзал.

Слово «найвартісніші» в даних не записане. Критерій ви побудуєте самі —
і захистите його.

Потім те саме для **тридцяти** об'єктів. Результат цієї, більшої задачі
стане базою порівняння в **лабораторній роботі № 8**, де ту саму задачу
розв'язуватимуть генетичним та мурашиним алгоритмами.

### Що здаєте

1. Цей ноутбук із виконаними комірками (усі виводи збережені).
2. Файл `lr5_результати.json` — результати для ЛР8.
3. Карту `lr5_маршрут.html`.
4. Звіт у кінці ноутбука: відповіді на сім питань.

### Що оцінюється

| критерій | 0 | 2 | 3 | 4 |
|---|---|---|---|---|
| Відбір об'єктів | теги взято навмання | фільтр працює | фільтр обґрунтовано | показано, що відкинуто і чому |
| Критерій рейтингу | немає | є формула | ваги пояснено | порівняно з альтернативним критерієм |
| Маршрут | не будується | будується | перевірено коректність | пояснено різницю методів числом |
| Карта | немає | точки на карті | маршрут по ребрах графа | видно різницю з прямими відрізками |
| Звіт | немає | формально | з числами | з висновком, якого не було в завданні |

---
## 0. Ваш варіант

Впишіть прізвище та номер у списку групи. Місто підтягнеться само.

In [1]:
STUDENT_NAME = 'Кшик Олена'    
LIST_NUMBER  = 9

CITIES = [
    'Львів', 'Київ', 'Харків', 'Одеса', 'Дніпро',
    'Чернівці', 'Івано-Франківськ', 'Тернопіль', 'Ужгород', 'Луцьк',
    'Рівне', 'Хмельницький', 'Вінниця', 'Житомир', 'Чернігів',
    'Черкаси', 'Полтава', 'Суми', 'Кропивницький', 'Запоріжжя',
    'Миколаїв', 'Кривий Ріг', 'Кам\u2019янець-Подільський', 'Мукачево',
    'Коломия', 'Дрогобич', 'Біла Церква', 'Ніжин', 'Остріг', 'Умань',
]

assert STUDENT_NAME and LIST_NUMBER > 0, 'впишіть прізвище та номер у списку'
CITY = CITIES[(LIST_NUMBER - 1) % len(CITIES)] + ', Україна'
print('варіант %d: %s' % (LIST_NUMBER, CITY))

варіант 9: Ужгород, Україна


> Якщо викладач призначив вам інше місто — просто перепишіть `CITY` вручну
> і зазначте це у звіті.

In [2]:
import importlib.util, subprocess, sys
REQUIRED = ['osmnx', 'geopandas', 'shapely', 'pyproj', 'pyarrow',
            'folium', 'mapclassify', 'h3', 'networkx']
missing = [p for p in REQUIRED if importlib.util.find_spec(p) is None]
if missing:
    print('встановлюю:', ', '.join(missing), '...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
print('усі пакети на місці')

усі пакети на місці


In [3]:
import warnings; warnings.filterwarnings('ignore')
import json, time, os, collections
import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
import matplotlib.pyplot as plt
import folium
from xyzservices import providers as xyz

BASEMAP = xyz.Esri.WorldGrayCanvas   # CARTO з 2026 р. вимагає ключ API
pd.set_option('display.width', 200)
print('geopandas', gpd.__version__)

geopandas 1.2.0


---
## 1. Два способи дістати дані

Робоча програма вимагає порівняти два способи доступу **за обсягом переданих
даних і за часом**:

* **живий запит** — `osmnx` питає Overpass API просто зараз;
* **знімок** — ті самі дані, збережені у файли GeoParquet, читаються з диска.

Нижче — живий запит. Запустіть його **один раз**: він збереже теку зі знімком,
і всі наступні запуски читатимуть з неї.

> Overpass — безкоштовний сервіс і обмежує частоту. Якщо запит не пройшов —
> зачекайте хвилину й повторіть. Якщо не проходить взагалі, візьміть
> резервний знімок у викладача і вкажіть це у звіті.

In [4]:
import osmnx as ox

OUT = 'osm_' + CITY.split(',')[0].strip().lower().replace(' ', '_')
os.makedirs(OUT, exist_ok=True)
ox.settings.use_cache = True
ox.settings.cache_folder = os.path.join(OUT, 'cache')
ox.settings.requests_timeout = 300
ox.settings.overpass_rate_limit = True

TOURISM_OK = ['museum', 'attraction', 'gallery', 'artwork', 'viewpoint']

def cache_bytes():
    import pathlib
    p = pathlib.Path(ox.settings.cache_folder)
    return sum(f.stat().st_size for f in p.glob('*.json')) if p.exists() else 0

print('тека знімка:', OUT)

тека знімка: osm_ужгород


### Дві налаштовані дрібниці, без яких запит не пройде

Це не формальності — кожна з них ламає збір по-своєму, і жодна не схожа
на свою справжню причину.

**Представитись.** Overpass відповідає **HTTP 406** клієнтові, який прийшов
із типовим заголовком `python-requests`. `osmnx` надсилає свій `User-Agent`
сам, але якщо ви звертаєтесь до сервісу напряму — надсилайте теж.
Рядок має бути **лише латиницею**: HTTP-заголовки передаються в latin-1,
і кирилиця кладе запит помилкою `UnicodeEncodeError` ще до відправлення.

**Обрати машину.** За іменем `overpass-api.de` стоїть кілька серверів, і
частина з них відмовляє у з’єднанні. `osmnx` закріплює одну адресу на
всю сесію — навмисно, щоб не порушувати облік слотів. Якщо закріпиться та,
що відмовляє, впаде весь збір. Тому адресу обираємо самі, вже перевірену.

In [5]:
import socket

USER_AGENT = ('LvivPolytechnic-SK26-coursework/1.0 '
              '(https://lpnu.ua; student lab assignment)')
assert USER_AGENT.isascii(), 'HTTP-заголовки — latin-1, кирилиця тут не працює'
ox.settings.http_user_agent = USER_AGENT


def pin_working_ip(hostname, port=443, timeout=8):
    """Закріпити для цього імені адресу, яка справді відповідає."""
    try:
        infos = socket.getaddrinfo(hostname, port, socket.AF_INET, socket.SOCK_STREAM)
    except Exception as e:
        print('не вдалось резолвити %s: %s' % (hostname, type(e).__name__))
        return None
    ips = sorted({i[4][0] for i in infos})
    good = []
    for ip in ips:
        s = socket.socket(); s.settimeout(timeout)
        try:
            s.connect((ip, port)); good.append(ip)
        except Exception:
            pass
        finally:
            s.close()
    print('%s: машин %d, відповідають %d %s' % (hostname, len(ips), len(good), good))
    if not good:
        return None
    original = socket.gethostbyname
    socket.gethostbyname = lambda h: good[0] if h == hostname else original(h)
    print('закріплено', good[0])
    return good[0]


pin_working_ip('overpass-api.de')

overpass-api.de: машин 2, відповідають 2 ['162.55.144.139', '65.109.112.52']
закріплено 162.55.144.139


'162.55.144.139'

> Якщо жодна машина не відповіла — виконайте комірку ще раз за хвилину,
> або візьміть у викладача резервний знімок і зазначте це у звіті.
> Інші дзеркала перелічені на сторінці
> [Overpass API](https://wiki.openstreetmap.org/wiki/Overpass_API);
> адресу можна підставити у `ox.settings.overpass_url`.

### Завдання 1.1

Допишіть функцію `fetch`: вона має зберегти шар у `.parquet`, а якщо файл
уже є — прочитати його, **не звертаючись до мережі**. Поверніть також,
скільки МБ пішло мережею і скільки секунд це зайняло.

Підказка щодо `clean`: у OSM одна колонка може містити і число, і список
(ребро, склеєне з кількох шляхів). Parquet вимагає один тип на колонку,
тому неоднорідну колонку треба звести до тексту **цілком**.

In [6]:
def clean(gdf):
    g = gdf.reset_index()
    for c in g.columns:
        if c == 'geometry':
            continue
        boxed = g[c].map(lambda v: isinstance(v, (list, dict, set, tuple))).any()
        if boxed or g[c].dtype == object:
            g[c] = g[c].map(
                lambda v: None if v is None or (isinstance(v, float) and pd.isna(v))
                else (json.dumps(v, ensure_ascii=False)
                      if isinstance(v, (list, dict, set, tuple)) else str(v))
            ).astype('string')
    return g


COST = {}

def fetch(name, getter):
    """Повертає шар. Якщо файл уже збережено — читає з диска."""
    path = os.path.join(OUT, name + '.parquet')
    if os.path.exists(path) and os.path.getsize(path) > 0:
        t0 = time.time()
        g = gpd.read_parquet(path)
        COST[name] = {'спосіб': 'знімок', 'записів': len(g),
                      'передано_МБ': 0.0,
                      'файл_МБ': round(os.path.getsize(path) / 1e6, 3),
                      'секунд': round(time.time() - t0, 3)}
        return g

    t0, b0 = time.time(), cache_bytes()
    clean(getter()).to_parquet(path, index=False)
    g = gpd.read_parquet(path)
    COST[name] = {'спосіб': 'живий запит', 'записів': len(g),
                  'передано_МБ': round((cache_bytes() - b0) / 1e6, 3),
                  'файл_МБ': round(os.path.getsize(path) / 1e6, 3),
                  'секунд': round(time.time() - t0, 1)}
    return g

In [7]:
boundary = fetch('boundary', lambda: ox.geocode_to_gdf(CITY))
poly = boundary.geometry.iloc[0]
print('межа міста отримана, площа %.1f км²'
      % (boundary.to_crs(32635).area.iloc[0] / 1e6))

межа міста отримана, площа 37.7 км²


In [8]:
heritage = fetch('heritage', lambda: ox.features_from_polygon(
    poly, {'historic': True, 'tourism': TOURISM_OK}))
stations = fetch('stations', lambda: ox.features_from_polygon(
    poly, {'railway': ['station', 'halt'], 'building': ['train_station']}))
amenities = fetch('amenities', lambda: ox.features_from_polygon(
    poly, {'amenity': ['cafe', 'restaurant', 'drinking_water', 'toilets']}))

NODES_P = os.path.join(OUT, 'road_nodes.parquet')
EDGES_P = os.path.join(OUT, 'road_edges.parquet')

if os.path.exists(NODES_P) and os.path.exists(EDGES_P):
    t0 = time.time()
    nodes, edges = gpd.read_parquet(NODES_P), gpd.read_parquet(EDGES_P)
    COST['мережа walk'] = {'спосіб': 'знімок', 'записів': len(nodes) + len(edges),
                           'передано_МБ': 0.0,
                           'файл_МБ': round((os.path.getsize(NODES_P)
                                             + os.path.getsize(EDGES_P)) / 1e6, 3),
                           'секунд': round(time.time() - t0, 3)}
else:
    t0, b0 = time.time(), cache_bytes()
    G_raw = ox.graph_from_polygon(poly, network_type='walk')
    nodes, edges = ox.convert.graph_to_gdfs(G_raw)
    clean(nodes).to_parquet(NODES_P, index=False)
    clean(edges).to_parquet(EDGES_P, index=False)
    nodes, edges = gpd.read_parquet(NODES_P), gpd.read_parquet(EDGES_P)
    COST['мережа walk'] = {'спосіб': 'живий запит', 'записів': len(nodes) + len(edges),
                           'передано_МБ': round((cache_bytes() - b0) / 1e6, 3),
                           'файл_МБ': round((os.path.getsize(NODES_P)
                                             + os.path.getsize(EDGES_P)) / 1e6, 3),
                           'секунд': round(time.time() - t0, 1)}

print('вузлів %d | ребер %d' % (len(nodes), len(edges)))

вузлів 7777 | ребер 20230


### Завдання 1.2

Зведіть `COST` у таблицю і дайте відповідь у звіті: **у скільки разів
читання знімка швидше за живий запит** і **скільки мегабайтів ви не передали
мережею** при повторному запуску.

Контрольна ознака правильності: при **другому** запуску ноутбука всі рядки
мають мати спосіб «знімок», а `передано_МБ` — нуль.

In [9]:
LIVE_P = os.path.join(OUT, 'cost_live.json')
live = {}
if os.path.exists(LIVE_P):
    with open(LIVE_P, encoding='utf-8') as f:
        live = json.load(f)
fresh = {k: v for k, v in COST.items() if v['спосіб'] == 'живий запит'}
if fresh:
    live.update(fresh)
    with open(LIVE_P, 'w', encoding='utf-8') as f:
        json.dump(live, f, ensure_ascii=False, indent=1)

display(pd.DataFrame(COST).T)

snap = {k: v for k, v in COST.items() if v['спосіб'] == 'знімок' and k in live}
if snap:
    cmp = pd.DataFrame({
        'живий_с':        {k: live[k]['секунд'] for k in snap},
        'знімок_с':       {k: snap[k]['секунд'] for k in snap},
        'не_передано_МБ': {k: live[k]['передано_МБ'] for k in snap},
    })
    cmp.loc['разом'] = cmp.sum()
    cmp['швидше_разів'] = (cmp['живий_с'] / cmp['знімок_с'].clip(lower=1e-3)).round(0)
    display(cmp)
    print('знімок швидший у %.0f разів; мережею не передано %.2f МБ'
          % (cmp.loc['разом', 'швидше_разів'], cmp.loc['разом', 'не_передано_МБ']))
else:
    print('живий запуск: вартість збережено у', LIVE_P)
    print('перезапустіть ноутбук — тоді з\u2019явиться порівняння')


,спосіб,записів,передано_МБ,файл_МБ,секунд
boundary,знімок,1,0.0,0.029,0.106
heritage,знімок,206,0.0,0.105,0.098
stations,знімок,7,0.0,0.043,0.069
amenities,знімок,256,0.0,0.076,0.075
мережа walk,знімок,28007,0.0,1.493,0.159


,живий_с,знімок_с,не_передано_МБ,швидше_разів
boundary,1.8,0.106,0.020,17.0
heritage,107.4,0.098,0.178,1096.0
stations,99.9,0.069,0.014,1448.0
amenities,69.2,0.075,0.082,923.0
мережа walk,82.6,0.159,3.907,519.0
разом,360.9,0.507,4.201,712.0


знімок швидший у 712 разів; мережею не передано 4.20 МБ


> **Чому пішохідна мережа, а не автомобільна.** У старому місті музеї стоять
> за сотню метрів один від одного, а автомобільних вузлів там одиниці. Кілька
> пам'яток сядуть на **один** вузол графа, і матриця віддалей скаже, що між
> ними нуль метрів. Задача розв'яжеться, відповідь буде неправильною, і
> повідомлення про помилку не з'явиться. У кроці 4 ви це перевірите явно.

---
## 2. Хто претендує на звання пам'ятки

У OpenStreetMap немає тега «визначна пам'ятка». Є два сімейства:

* `historic=*` — `castle`, `church`, `monument`, `memorial`, `ruins`, `tower`, `yes`…
* `tourism=*` — `museum`, `gallery`, `attraction`, `artwork`, `viewpoint`

Перетин неповний: музей у сучасній будівлі має лише `tourism`, руїни фортеці —
лише `historic`. Тому беремо **об'єднання**.

📖 <https://wiki.openstreetmap.org/wiki/Map_Features>

### Завдання 2.1

Відберіть кандидатів і **назвіть у звіті, скільки об'єктів ви відкинули й чому**.

In [10]:
by_historic = heritage['historic'].notna()
by_tourism  = heritage['tourism'].isin(TOURISM_OK)
mask  = by_historic | by_tourism           
named = heritage['name'].notna()

cand = heritage[mask & named].reset_index(drop=True)

print('усього у шарі heritage :', len(heritage))
print('  historic             :', int(by_historic.sum()))
print('  tourism              :', int(by_tourism.sum()))
print('  обидва одразу        :', int((by_historic & by_tourism).sum()))
print('підходять за тегами    :', int(mask.sum()))
print('з них мають name       :', len(cand))

dropped = heritage[mask & ~named]
kind = dropped['historic'].fillna(dropped['tourism'])
print()
print('відкинуто без назви: %d' % len(dropped))
print(kind.value_counts().to_string())


усього у шарі heritage : 206
  historic             : 104
  tourism              : 117
  обидва одразу        : 15
підходять за тегами    : 206
з них мають name       : 170

відкинуто без назви: 36
historic
memorial          10
wayside_cross      8
citywalls          7
viewpoint          4
city_gate          2
artwork            2
wayside_shrine     1
building           1
yes                1


In [11]:
assert 'cand' in dir(), 'спершу виконайте завдання 2.1'
print('=== historic ===');  print(cand['historic'].value_counts().head(10).to_string())
print()
print('=== tourism ===');   print(cand['tourism'].value_counts().head(10).to_string())
print()
print('типи геометрії:', cand.geometry.geom_type.value_counts().to_dict())

=== historic ===
historic
memorial         39
building         19
church            6
tank              3
tomb              2
ruins             2
wayside_cross     1
castle            1
manor             1

=== tourism ===
tourism
artwork       82
museum        22
attraction     4
gallery        2
viewpoint      1

типи геометрії: {'Point': 133, 'Polygon': 37}


> **Перевірте себе.** Серед типів геометрії мають бути і `Point`, і `Polygon`.
> Якщо самі лише полігони — ви відфільтрували зайве: пам'ятники, меморіали та
> оглядові точки в OSM є **вузлами**, у них немає будівлі.

### Якщо кандидатів менше тридцяти

Невелике місто може дати 15–20 об'єктів. Тоді:

1. розширте перелік `tourism` (наприклад, додайте `theatre`, `place_of_worship`);
2. або зменште `N_LARGE` до того, що є;

і **обов'язково зазначте це у звіті** разом із числом.

---
## 3. Критерій «найкращі»

Ключовий крок роботи. Стовпця «цінність» у даних немає. Його треба
сконструювати і захистити.

Сигнали, які є в OSM:

| сигнал | що означає |
|---|---|
| `wikidata` / `wikipedia` | про об'єкт є стаття — хтось сторонній визнав його вартим опису |
| кількість колонок `name:*` | назву переклали N мовами — перекладають те, що шукають іноземці |
| `tourism=museum/gallery` | призначений для відвідування |
| `historic` заповнений | має історичну природу |
| площа полігона | розмір — **найслабший сигнал**: великим буває і склад |

### Завдання 3.1

Побудуйте числовий рейтинг. Вимоги:

* використати **щонайменше три** різні сигнали;
* звести різнорідні величини до спільної шкали (підказка: `rank(pct=True)`);
* **пояснити у звіті, чому саме такі ваги**.

In [12]:
metric = cand.to_crs(32635)
cand['area_m2'] = np.where(metric.geometry.geom_type.isin(['Polygon', 'MultiPolygon']),
                           metric.area.values, 0.0)

# ----- ВАШ КОД -----
cand['wiki'] = 0.5 * cand['wikidata'].notna() + 0.5 * cand['wikipedia'].notna()

lang_cols = [c for c in cand.columns if c.startswith('name:') and c != 'name:uk']
cand['n_lang'] = cand[lang_cols].notna().sum(axis=1)

cand['museum']  = cand['tourism'].isin(['museum', 'gallery']).astype(float)
cand['is_hist'] = cand['historic'].notna().astype(float)

signals = pd.DataFrame({
    'wiki':    cand['wiki'],
    'n_lang':  cand['n_lang'].rank(pct=True),
    'museum':  cand['museum'],
    'is_hist': cand['is_hist'],
    'area':    cand['area_m2'].rank(pct=True),
})
WEIGHTS = {'wiki': 0.35, 'n_lang': 0.25, 'museum': 0.20, 'is_hist': 0.15, 'area': 0.05}
assert abs(sum(WEIGHTS.values()) - 1) < 1e-9
cand['score'] = signals.mul(pd.Series(WEIGHTS)).sum(axis=1)

print('кандидатів           :', len(cand))
print('мають wikidata       :', int(cand['wikidata'].notna().sum()))
print('мають статтю         :', int(cand['wikipedia'].notna().sum()))
print('name:uk = name       :', int((cand['name:uk'] == cand['name']).sum()))
print('перекладів назви     :', cand['n_lang'].value_counts().sort_index().to_dict())
print('музеїв і галерей     :', int(cand['museum'].sum()))
print('historic заповнено   :', int(cand['is_hist'].sum()))
print('полігонів (є площа)  :', int((cand['area_m2'] > 0).sum()))

кандидатів           : 170
мають wikidata       : 36
мають статтю         : 25
name:uk = name       : 165
перекладів назви     : {0: 79, 1: 52, 2: 30, 3: 6, 6: 1, 8: 1, 10: 1}
музеїв і галерей     : 24
historic заповнено   : 74
полігонів (є площа)  : 37


### Завдання 3.2 — обов'язкове порівняння

Побудуйте **другий** рейтинг за найпростішим критерієм — площею будівлі —
і порахуйте, скільки об'єктів збігається в перших десяти.

Це головне питання роботи: обидва списки отримані з тих самих даних, тим
самим кодом, без жодної помилки. Якщо вони різні — поясніть у звіті, що це
означає для слова «найкращі».

In [13]:
N_SMALL, N_LARGE = 10, 30

# ----- ВАШ КОД -----
top_score = cand.nlargest(N_LARGE, 'score').reset_index(drop=True)
top10     = top_score.head(N_SMALL)
top_naive = cand.nlargest(N_SMALL, 'area_m2').reset_index(drop=True)
common    = set(top10['name']) & set(top_naive['name'])
overlap   = len(common)
print('перетин: %d з %d' % (overlap, N_SMALL))

cmp = pd.DataFrame({'за критерієм': top10['name'],
                    'score': top10['score'].round(3),
                    'за площею': top_naive['name'],
                    'площа, м²': top_naive['area_m2'].round().astype(int)})
cmp.index += 1
display(cmp)

cols = ['name', 'historic', 'tourism', 'area_m2', 'wiki', 'n_lang', 'score']
print('лише за площею:')
display(top_naive.loc[~top_naive['name'].isin(common), cols].round(3))
print('лише за критерієм:')
display(top10.loc[~top10['name'].isin(common), cols].round(3))

перетин: 7 з 10


,за критерієм,score,за площею,"площа, м²"
1,Закарпатський краєзнавчий музей ім. Тиводара Л...,0.815,Закарпатський краєзнавчий музей ім. Тиводара Л...,24136
2,Ужгородський замок,0.798,Ужгородський замок,16171
3,Хресто-Воздвиженський катедральний собор,0.798,Липова алея,12253
4,Костел святого Юрія,0.795,Будинок правління Ужанського комітату,2239
5,Горянська ротонда Св. Анни,0.788,Совине Гніздо,1477
6,Совине Гніздо,0.764,Резиденція єпископа,1287
7,Резиденція єпископа,0.764,Костел святого Юрія,843
8,Реформатська Церква,0.763,Хресто-Воздвиженський катедральний собор,646
9,Тарас Шевченко,0.762,Церква святого Юрія,487
10,Обласний художній музей ім. Йосипа Бокшая,0.725,Реформатська Церква,426


лише за площею:


,name,historic,tourism,area_m2,wiki,n_lang,score
2,Липова алея,NaN,attraction,12252.874,1.0,1,0.555
3,Будинок правління Ужанського комітату,building,NaN,2239.374,1.0,0,0.608
8,Церква святого Юрія,ruins,NaN,486.877,1.0,1,0.703


лише за критерієм:


,name,historic,tourism,area_m2,wiki,n_lang,score
4,Горянська ротонда Св. Анни,church,attraction,239.68,1.0,3,0.788
8,Тарас Шевченко,memorial,NaN,0.00,1.0,3,0.762
9,Обласний художній музей ім. Йосипа Бокшая,NaN,museum,0.00,1.0,1,0.725


### Завдання 3.3 — дублікати

OSM наповнюють різні люди. Той самий музей трапляється двічі під різними
назвами, а вежа буває окремим об'єктом усередині палацу. Знайдіть у `top_score`
пари, що стоять ближче **60 метрів** одна до одної, і вирішіть, що з ними робити.

Рішення (лишити / злити / видалити) опишіть у звіті.

In [14]:
def close_pairs(gdf, limit_m=60):
    p = gdf.to_crs(32635).geometry.representative_point()
    XY = np.c_[p.x.values, p.y.values]
    D = np.sqrt(((XY[:, None, :] - XY[None, :, :]) ** 2).sum(-1))
    np.fill_diagonal(D, np.inf)
    out = [(i, j, D[i, j]) for i in range(len(XY)) for j in range(i + 1, len(XY))
           if D[i, j] < limit_m]
    return sorted(out, key=lambda t: t[2])

def nested_pairs(gdf):
    """Пари (i, j): точка об'єкта i лежить усередині полігона об'єкта j."""
    m = gdf.to_crs(32635)
    hit = gpd.sjoin(gpd.GeoDataFrame(geometry=m.geometry.representative_point()),
                    m[m.geom_type.isin(['Polygon', 'MultiPolygon'])][['geometry']],
                    predicate='within')
    return {(i, j) for i, j in hit['index_right'].items() if i != j}

# ----- ВАШ КОД -----
SAME = 'злити'
KEEP = 'лишити'
MUSEUM = 'Закарпатський краєзнавчий музей ім. Тиводара Легоцького'
ART    = 'Обласний художній музей ім. Йосипа Бокшая'
DECISIONS = {
    (MUSEUM, 'Ужгородський замок'):   (SAME, 'музей міститься в стінах замку'),
    (MUSEUM, 'Церква святого Юрія'):  (SAME, 'руїни церкви стоять на території замку'),
    ('Ужгородський замок', 'Церква святого Юрія'): (SAME, 'руїни на території замку'),
    (ART, 'Будинок правління Ужанського комітату'):
        (SAME, 'у OSM це будівля і її точка-«музей»: одне місце'),
    ('Т-70', 'Т-34-85'):              (SAME, 'один меморіальний майданчик, два постаменти'),
    ('Хресто-Воздвиженський катедральний собор', 'Резиденція єпископа'):
        (KEEP, 'різні споруди: собор і палац'),
    ('Реформатська Церква', ART):      (KEEP, 'різні споруди: церква і музей'),
    ('Реформатська Церква', 'Будинок правління Ужанського комітату'):
        (KEEP, 'різні споруди: церква і будинок комітату'),
    ('Августин Волошин', 'Липова алея'):
        (KEEP, 'пам’ятник стоїть у парку, але це окремий об’єкт'),
}
DECISIONS.update({(b, a): v for (a, b), v in list(DECISIONS.items())})
TITLES = {MUSEUM: 'Ужгородський замок (краєзнавчий музей)',
          ART: 'Будинок Ужанського комітату (обласний художній музей)',
          'Т-70': 'Танки-пам’ятники Т-70 і Т-34-85', 'Т-34-85': 'Танки-пам’ятники Т-70 і Т-34-85'}

def build_top(by, n):
    pool, absorbed, seen, rows = cand.copy(), {}, set(), []
    while True:
        top = pool.nlargest(n, by).reset_index(drop=True)
        found = {}                                   
        for i, j, d in close_pairs(top):
            found[frozenset((top['name'][i], top['name'][j]))] = d
        for i, j in nested_pairs(top):
            found[frozenset((top['name'][i], top['name'][j]))] = 0.0
        found = {k: d for k, d in found.items() if k not in seen}
        if not found:
            return top, rows, absorbed
        into = {}
        for pair, d in sorted(found.items(), key=lambda t: t[1]):
            seen.add(pair)
            a, b = sorted(pair, key=lambda x: top.index[top['name'] == x][0])
            verdict, why = DECISIONS[(a, b)]
            rows.append({'A': a, 'B': b, 'відстань, м': round(d, 1), 'рішення': verdict, 'чому': why})
            if verdict == SAME:                        
                a, b = sorted((into.get(a, a), into.get(b, b)), key=lambda x: top.index[top['name'] == x][0])
                if a != b:
                    into[b] = a
                    absorbed.setdefault(a, []).extend([b] + absorbed.pop(b, []))
        pool = pool[~pool['name'].isin(into)]

top_score, rows, absorbed = build_top('score', N_LARGE)
top_score['name_osm'] = top_score['name']
top_score['злито'] = [', '.join(absorbed.get(n, [])) for n in top_score['name']]
top_score['name'] = top_score['name'].replace(TITLES)
top10 = top_score.head(N_SMALL)                 # мала задача — вже без дублікатів

pairs = pd.DataFrame(rows)
print('пар ближче 60 м у top_score: %d' % len(pairs))
display(pairs)
print('об’єкти, що поглинули дублікати:')
display(top_score.loc[top_score['злито'] != '', ['name', 'name_osm', 'злито']])


пар ближче 60 м у top_score: 9


,A,B,"відстань, м",рішення,чому
0,Обласний художній музей ім. Йосипа Бокшая,Будинок правління Ужанського комітату,0.0,злити,у OSM це будівля і її точка-«музей»: одне місце
1,Закарпатський краєзнавчий музей ім. Тиводара Л...,Ужгородський замок,0.0,злити,музей міститься в стінах замку
2,Ужгородський замок,Церква святого Юрія,0.0,злити,руїни на території замку
3,Августин Волошин,Липова алея,0.0,лишити,"пам’ятник стоїть у парку, але це окремий об’єкт"
4,Закарпатський краєзнавчий музей ім. Тиводара Л...,Церква святого Юрія,0.0,злити,руїни церкви стоять на території замку
5,Т-70,Т-34-85,12.7,злити,"один меморіальний майданчик, два постаменти"
6,Хресто-Воздвиженський катедральний собор,Резиденція єпископа,32.6,лишити,різні споруди: собор і палац
7,Реформатська Церква,Обласний художній музей ім. Йосипа Бокшая,46.4,лишити,різні споруди: церква і музей
8,Реформатська Церква,Будинок правління Ужанського комітату,49.5,лишити,різні споруди: церква і будинок комітату


об’єкти, що поглинули дублікати:


,name,name_osm,злито
0,Ужгородський замок (краєзнавчий музей),Закарпатський краєзнавчий музей ім. Тиводара Л...,"Ужгородський замок, Церква святого Юрія"
8,Будинок Ужанського комітату (обласний художній...,Обласний художній музей ім. Йосипа Бокшая,Будинок правління Ужанського комітату
19,Танки-пам’ятники Т-70 і Т-34-85,Т-70,Т-34-85


In [15]:
# 3.2 ще раз, після злиття: одне місце рахуємо один раз в обох рейтингах
groups = []
for (a, b), (verdict, _) in DECISIONS.items():
    if verdict == SAME:
        hit = [s for s in groups if a in s or b in s]
        groups = [s for s in groups if s not in hit] + [set().union({a, b}, *hit)]
place = {n: frozenset(s) for s in groups for n in s}
key = lambda n: place.get(n, n)

top_naive_d, _, _ = build_top('area_m2', N_SMALL)
k_score = {key(n): t for n, t in zip(top10['name_osm'], top10['name'])}
k_naive = {key(n): n for n in top_naive_d['name']}
common_d = k_score.keys() & k_naive.keys()
print('перетин після злиття дублікатів: %d з %d (до злиття було %d)'
      % (len(common_d), N_SMALL, overlap))
print('лише за площею  :', ', '.join(v for k, v in k_naive.items() if k not in common_d))
print('лише за критерієм:', ', '.join(v for k, v in k_score.items() if k not in common_d))


перетин після злиття дублікатів: 8 з 10 (до злиття було 7)
лише за площею  : Липова алея, Будинок Іштвана Дєндеші
лише за критерієм: Горянська ротонда Св. Анни, Тарас Шевченко


---
## 4. Старт: вокзал, і прив'язка точок до графа

In [16]:
print('станції у шарі:', len(stations))
print(stations[['name', 'railway', 'building']].dropna(how='all').head(12).to_string())

станції у шарі: 7
             name  railway       building
0  260 км (ЛВЧ-5)  station            NaN
1            Парк  station            NaN
2       Молодіжна  station            NaN
3         Ужгород  station            NaN
4  Вокзал Ужгород      NaN  train_station
5             NaN      NaN  train_station
6             NaN      NaN  train_station


In [17]:
station = stations[(stations['railway'] == 'station') & (stations['name'] == 'Ужгород')].head(1)
print('старт:', station['name'].iloc[0])

старт: Ужгород


In [18]:
G = nx.DiGraph()
for t in nodes.itertuples():
    G.add_node(t.osmid, x=t.x, y=t.y)
for t in edges.itertuples():
    L = float(t.length)
    if G.has_edge(t.u, t.v):
        G[t.u][t.v]['length'] = min(G[t.u][t.v]['length'], L)
    else:
        G.add_edge(t.u, t.v, length=L)
print('граф: %d вузлів, %d ребер' % (G.number_of_nodes(), G.number_of_edges()))

NODES_M = nodes.to_crs(32635)
NODE_XY = np.c_[NODES_M.geometry.x.values, NODES_M.geometry.y.values]
NODE_ID = nodes['osmid'].values

def snap(points_gdf):
    pmet = points_gdf.to_crs(32635)
    ids, dist = [], []
    for p in pmet.geometry:
        d2 = (NODE_XY[:, 0] - p.x) ** 2 + (NODE_XY[:, 1] - p.y) ** 2
        k = int(np.argmin(d2))
        ids.append(NODE_ID[k]); dist.append(float(np.sqrt(d2[k])))
    return np.array(ids), np.array(dist)

def make_points(top):
    names = ['СТАРТ: ' + str(station['name'].iloc[0])] + list(top['name'])
    geoms = [station.geometry.iloc[0].representative_point()] + \
            [g.representative_point() for g in top.geometry]
    return gpd.GeoDataFrame({'name': names}, geometry=geoms, crs='EPSG:4326')

граф: 7777 вузлів, 20035 ребер


### Завдання 4.1 — обов'язкова перевірка

Напишіть `check_snap`, яка повідомляє:

* скільки точок і скільки **різних** вузлів вони зайняли;
* медіану й максимум відстані «точка → вузол»;
* якщо вузлів менше, ніж точок — **назви точок, що злиплись**.

**Контрольна ознака:** на пішохідній мережі різних вузлів має бути **стільки ж,
скільки точок**. Якщо менше — ви взяли автомобільну мережу або місто надто мале;
зафіксуйте це у звіті, бо далі такі пари отримають нульову віддаль.

In [19]:
def check_snap(points, ids, dist, label):
    # ----- ВАШ КОД -----
    n, uniq = len(points), len(set(ids))
    print('[%s] точок: %d | різних вузлів: %d' % (label, n, uniq))
    print('  відстань точка → вузол: медіана %.1f м, максимум %.1f м'
          % (np.median(dist), np.max(dist)))
    if uniq < n:
        by_node = collections.defaultdict(list)
        for name, node in zip(points['name'], ids):
            by_node[node].append(name)
        print('  точки, що злиплись (%d зайвих):' % (n - uniq))
        for node, names in by_node.items():
            if len(names) > 1:
                print('   вузол %s: %s' % (node, ' | '.join(names)))
    else:
        print('  ок: кожна точка має власний вузол')
    return uniq == n


---
## 5. Матриця віддалей

Дві матриці: **пряма лінія** і **дорогою**. Їхнє відношення — коефіцієнт
обходу — показує, наскільки пряма лінія бреше.

### Завдання 5.1

In [20]:
def road_matrix(ids):
    k = len(ids)
    D = np.full((k, k), np.inf)
    for i, s in enumerate(ids):
        lengths = nx.single_source_dijkstra_path_length(G, s, weight='length')
        for j, t in enumerate(ids):
            D[i, j] = lengths.get(t, np.inf)
    np.fill_diagonal(D, 0.0)
    return D

def straight_matrix(points):
    pm = points.to_crs(32635).geometry
    XY = np.c_[pm.x.values, pm.y.values]
    return np.sqrt(((XY[:, None, :] - XY[None, :, :]) ** 2).sum(-1))

def detour_factor(D, E):
    off = ~np.eye(len(D), dtype=bool)
    ok = off & np.isfinite(D) & (E > 0)
    return float(np.median(D[ok] / E[ok]))

**Контрольні ознаки — звіртесь, не підганяючи:**

* коефіцієнт обходу для пішохідної мережі у місті зазвичай **1,2–1,6**;
* якщо у вас рівно **1,0** — ви порівнюєте матрицю саму із собою;
* якщо **більше 2** — найімовірніше взято автомобільну мережу з односторонніми
  вулицями, або місто має річку/залізницю, яку нема де перетнути. Друге —
  цікавий результат, перше — помилка. Розберіться, що саме у вас, і напишіть це.

Якщо серед віддалей трапились `inf` — частина графа не зв'язана з рештою.
Не замінюйте `inf` великим числом мовчки: спершу з'ясуйте, які це точки.

In [21]:
pts10 = make_points(top10)
ids10, dist10 = snap(pts10)
check_snap(pts10, ids10, dist10, 'мала задача')

t0 = time.time(); D10 = road_matrix(ids10); print('матриця за %.3f с' % (time.time() - t0))
E10 = straight_matrix(pts10)
print('недосяжних пар    :', int(np.isinf(D10).sum()))
print('асиметрія         : %.0f м' % float(np.max(np.abs(D10 - D10.T))))
print('коефіцієнт обходу : %.2f' % detour_factor(D10, E10))

[мала задача] точок: 11 | різних вузлів: 11
  відстань точка → вузол: медіана 18.5 м, максимум 38.3 м
  ок: кожна точка має власний вузол
матриця за 0.404 с
недосяжних пар    : 0
асиметрія         : 0 м
коефіцієнт обходу : 1.34


> **Асиметрія.** $D_{ij} \neq D_{ji}$ — не помилка: односторонній рух робить
> шлях туди і назад різним. На пішохідній мережі асиметрія мала, на
> автомобільній — помітна. Подивіться на своє число і поясніть його у звіті.

---
## 6. Задача комівояжера

| метод | що робить | складність |
|---|---|---|
| найближчий сусід | щоразу йде в найближчу невідвідану точку | $O(n^2)$ |
| 2-opt | розвертає ділянки готового маршруту, поки коротшає | $O(n^2)$ за прохід |
| Held–Karp | динамічне програмування за підмножинами — **точний** | $O(2^n n^2)$ часу, $O(2^n n)$ пам'яті |

Held–Karp дано готовим: він не є метою роботи, але без нього ви не дізнаєтесь,
**наскільки** помиляються перші два.

### Завдання 6.1 — реалізуйте перші два методи

In [22]:
def tour_length(D, tour):
    return float(sum(D[tour[i], tour[i + 1]] for i in range(len(tour) - 1)))

def nearest_neighbour(D):
    k = len(D)
    tour, left = [0], set(range(1, k))
    while left:
        cur = tour[-1]
        nxt = min(left, key=lambda j: D[cur, j])
        tour.append(nxt); left.remove(nxt)
    return tour + [0]

def two_opt(D, tour):
    tour = list(tour)
    improved = True
    while improved:
        improved = False
        for i in range(1, len(tour) - 2):
            for j in range(i + 1, len(tour) - 1):
                new = tour[:i] + tour[i:j + 1][::-1] + tour[j + 1:]
                if tour_length(D, new) < tour_length(D, tour) - 1e-9:
                    tour = new
                    improved = True
    return tour

In [23]:
def held_karp(D):
    """Точний розв'язок. Вузол 0 — старт і фініш. ДАНО, писати не треба."""
    k = len(D); N = k - 1; INF = float('inf')
    dp = [[INF] * N for _ in range(1 << N)]
    par = [[-1] * N for _ in range(1 << N)]
    for j in range(N):
        dp[1 << j][j] = D[0, j + 1]
    for S in range(1 << N):
        for j in range(N):
            cur = dp[S][j]
            if cur == INF or not (S >> j) & 1:
                continue
            for t in range(N):
                if (S >> t) & 1:
                    continue
                S2 = S | (1 << t); v = cur + D[j + 1, t + 1]
                if v < dp[S2][t]:
                    dp[S2][t] = v; par[S2][t] = j
    full = (1 << N) - 1
    bj = min(range(N), key=lambda j: dp[full][j] + D[j + 1, 0])
    best = dp[full][bj] + D[bj + 1, 0]
    path = []; S = full; j = bj
    while j != -1:
        path.append(j + 1); pj = par[S][j]; S ^= (1 << j); j = pj
    return [0] + path[::-1] + [0], float(best)

### Самоперевірка

Ці твердження мають виконатись для **будь-якого** міста. Якщо хоч одне впало —
шукайте помилку в коді, а не в даних.

In [24]:
def check_tour(tour, k, label):
    assert tour[0] == 0 and tour[-1] == 0, '%s: маршрут має починатись і закінчуватись на старті' % label
    assert sorted(tour[:-1]) == list(range(k)), '%s: кожна точка рівно один раз' % label
    print('%s: структура маршруту правильна (%d точок)' % (label, k))

nn10 = nearest_neighbour(D10)
op10 = two_opt(D10, nn10)
ex10, L_opt = held_karp(D10)

for t, lbl in [(nn10, 'найближчий сусід'), (op10, '2-opt'), (ex10, 'Held-Karp')]:
    check_tour(t, len(ids10), lbl)

assert tour_length(D10, op10) <= tour_length(D10, nn10) + 1e-6, '2-opt не може бути гіршим за вхідний маршрут'
assert L_opt <= tour_length(D10, op10) + 1e-6, 'точний розв\u2019язок не може бути гіршим за наближений'
print()
print('усі перевірки пройдено')

найближчий сусід: структура маршруту правильна (11 точок)
2-opt: структура маршруту правильна (11 точок)
Held-Karp: структура маршруту правильна (11 точок)

усі перевірки пройдено


In [25]:
rows = []
for name, tour in [('найближчий сусід', nn10), ('2-opt', op10), ('Held-Karp (точний)', ex10)]:
    L = tour_length(D10, tour)
    rows.append({'метод': name, 'довжина, м': round(L),
                 'гірше оптимуму, %': round(100 * (L / L_opt - 1), 2)})
display(pd.DataFrame(rows).set_index('метод'))

print()
print('оптимальний порядок:')
for step, i in enumerate(ex10):
    print('  %2d. %s' % (step, pts10['name'].iloc[i]))
print()
print('довжина: %.2f км' % (L_opt / 1000))

,"довжина, м","гірше оптимуму, %"
метод,,
найближчий сусід,14431,6.93
2-opt,13568,0.53
Held-Karp (точний),13496,0.00



оптимальний порядок:
   0. СТАРТ: Ужгород
   1. Горянська ротонда Св. Анни
   2. Ужгородський замок (краєзнавчий музей)
   3. Резиденція єпископа
   4. Хресто-Воздвиженський катедральний собор
   5. Свято-Преображенський храм
   6. Реформатська Церква
   7. Будинок Ужанського комітату (обласний художній музей)
   8. Совине Гніздо
   9. Тарас Шевченко
  10. Костел святого Юрія
  11. СТАРТ: Ужгород

довжина: 13.50 км


### Завдання 6.2 — велика задача

Спершу оцініть, чи можливий тут точний розв'язок. Порахуйте для
$n = 11, 16, 21, 26, 31$ розмір таблиці Held–Karp ($2^{n-1}\cdot(n-1)$ клітин)
і скільки це гігабайтів при 8 байтах на клітину.

In [26]:
growth = []
for n in [11, 16, 21, 26, 31]:
    cells = 2 ** (n - 1) * (n - 1)
    growth.append({'n': n, 'клітин': cells, 'ГБ (8 Б/клітину)': round(cells * 8 / 1e9, 3)})
display(pd.DataFrame(growth).set_index('n'))

,клітин,ГБ (8 Б/клітину)
n,,
11,10240,0.000
16,491520,0.004
21,20971520,0.168
26,838860800,6.711
31,32212254720,257.698


Подивіться на останній рядок і напишіть у звіті, **чому саме** точний розв'язок
на 30 точках недоступний. Слово «довго» відповіддю не є.

In [27]:
pts30 = make_points(top_score)
ids30, dist30 = snap(pts30)
check_snap(pts30, ids30, dist30, 'велика задача')

D30 = road_matrix(ids30)
E30 = straight_matrix(pts30)
print('коефіцієнт обходу: %.2f' % detour_factor(D30, E30))

nn30 = nearest_neighbour(D30)
t0 = time.time(); op30 = two_opt(D30, nn30); t_op30 = time.time() - t0
check_tour(op30, len(ids30), '2-opt на 30 точках')

L_nn30, L_op30 = tour_length(D30, nn30), tour_length(D30, op30)
print()
print('найближчий сусід : %8.0f м' % L_nn30)
print('2-opt            : %8.0f м  за %.3f с  (коротше на %.1f%%)'
      % (L_op30, t_op30, 100 * (1 - L_op30 / L_nn30)))
print()
print('ЦЕ — база порівняння для лабораторної роботи № 8.')

[велика задача] точок: 31 | різних вузлів: 31
  відстань точка → вузол: медіана 17.9 м, максимум 118.5 м
  ок: кожна точка має власний вузол
коефіцієнт обходу: 1.32
2-opt на 30 точках: структура маршруту правильна (31 точок)

найближчий сусід :    20561 м
2-opt            :    19116 м  за 0.048 с  (коротше на 7.0%)

ЦЕ — база порівняння для лабораторної роботи № 8.


---
## 7. Просторова індексація: H3

H3 — ієрархічна сітка шестикутників. Рівень 9 — приблизно 200 м по ребру.

📖 <https://h3geo.org/docs/core-library/restable/>

### Завдання 7.1

Розкладіть **усіх** кандидатів по комірках H3 рівня 9 і побудуйте картограму
щільності. На тій самій карті покажіть межу міста і точки вашого маршруту.

In [29]:
import h3
from shapely.geometry import Polygon

H3_RES = 9
print('середнє ребро комірки: %.0f м' % h3.average_hexagon_edge_length(H3_RES, unit='m'))

cp = cand.geometry.representative_point()
cand['h3'] = [h3.latlng_to_cell(p.y, p.x, H3_RES) for p in cp]
dens = cand.groupby('h3').size().rename('n').reset_index()
hexes = gpd.GeoDataFrame(
    dens,
    geometry=[Polygon([(lng, lat) for lat, lng in h3.cell_to_boundary(c)]) for c in dens['h3']],
    crs='EPSG:4326')

srt = dens['n'].sort_values(ascending=False).to_numpy()
half = int(np.searchsorted(np.cumsum(srt), len(cand) / 2) + 1)
print('кандидатів: %d | комірок із пам’ятками: %d' % (len(cand), len(hexes)))
print('половина (%d з %d) — у %d найщільніших комірках (%.0f%% зайнятих)'
      % (int(np.ceil(len(cand) / 2)), len(cand), half, 100 * half / len(hexes)))
print('найщільніша комірка: %d пам’яток' % srt[0])

b = boundary.geometry.iloc[0].centroid
fmap7 = folium.Map(location=(b.y, b.x), zoom_start=13, tiles=BASEMAP)
folium.Choropleth(geo_data=hexes.to_json(), data=hexes, columns=['h3', 'n'],
                  key_on='feature.properties.h3', fill_color='YlOrRd',
                  fill_opacity=0.7, line_opacity=0.3, legend_name='пам’яток у комірці').add_to(fmap7)
folium.GeoJson(boundary.to_json(),
               style_function=lambda f: {'color': '#222', 'weight': 2, 'fill': False}).add_to(fmap7)
route_names = set(top10['name_osm']) | {n for v in top10['злито'] for n in v.split(', ') if n}
for name, g in zip(cand['name'], cp):
    if name not in route_names:
        folium.CircleMarker((g.y, g.x), radius=3, color='#999', weight=1, fill=True,
                            fill_color='#bbb', fill_opacity=0.8, tooltip=str(name)).add_to(fmap7)
for step, i in enumerate(ex10[:-1]):
    g = pts10.geometry.iloc[i]
    folium.Marker((g.y, g.x), tooltip='%d. %s' % (step, pts10['name'].iloc[i]),
                  icon=folium.DivIcon(html='<div style="background:#d62728;color:#fff;'
                  'border-radius:50%%;width:22px;height:22px;line-height:22px;'
                  'text-align:center;font-weight:bold">%s</div>' % ('С' if step == 0 else step),
                  icon_size=(22, 22), icon_anchor=(11, 11))).add_to(fmap7)

minx, miny, maxx, maxy = pts10.total_bounds
fmap7.fit_bounds([(miny, minx), (maxy, maxx)], padding=(30, 30))
fmap7.save('lr5_h3.html')
fmap7


середнє ребро комірки: 201 м
кандидатів: 170 | комірок із пам’ятками: 39
половина (85 з 170) — у 7 найщільніших комірках (18% зайнятих)
найщільніша комірка: 23 пам’яток


> **Пастка.** `h3.cell_to_boundary` повертає пари **(lat, lng)**, а
> `shapely.Polygon` чекає **(x, y)**, тобто (lng, lat). Переплутаєте — місто
> опиниться в Індійському океані. Це видно одразу, тому дивіться на карту.

### Питання до звіту

У скількох комірках зосереджено половину всіх пам'яток вашого міста?
Що це говорить про форму маршруту?

---
## 8. Просторове поєднання

### Завдання 8.1

Для кожної точки маршруту знайдіть найближчий заклад із шару `amenities`
(`sjoin_nearest`) і виведіть таблицю з відстанню в метрах.

Нагадування: `sjoin_nearest` рахує в одиницях CRS. У градусах результат буде
безглуздим — перепроєктуйте обидва шари в метричну систему.

In [30]:
am = amenities[['name', 'amenity', 'geometry']].copy()
for tag in ['name:uk', 'name:en', 'brand', 'operator']:
    if tag in amenities:
        am['name'] = am['name'].fillna(amenities[tag])
am['name'] = am['name'].fillna('(без назви)')
am['geometry'] = am.to_crs(32635).geometry.representative_point()
am = am.set_geometry('geometry', crs=32635)
route_pts = pts10.to_crs(32635).rename(columns={'name': 'пам’ятка'})

near = gpd.sjoin_nearest(route_pts, am.rename(columns={'name': 'заклад'}),
                         how='left', distance_col='відстань, м')
near = near.loc[~near.index.duplicated(), ['пам’ятка', 'заклад', 'amenity', 'відстань, м']]
near['відстань, м'] = near['відстань, м'].round().astype(int)
display(near.reset_index(drop=True))
print('закладів у шарі: %d | медіана відстані %.0f м, максимум %d м'
      % (len(am), near['відстань, м'].median(), near['відстань, м'].max()))


,пам’ятка,заклад,amenity,"відстань, м"
0,СТАРТ: Ужгород,Crazy Rocky,restaurant,179
1,Ужгородський замок (краєзнавчий музей),(без назви),restaurant,51
2,Хресто-Воздвиженський катедральний собор,Villa Alföld,restaurant,74
3,Костел святого Юрія,Byron,cafe,33
4,Горянська ротонда Св. Анни,(без назви),cafe,1010
5,Совине Гніздо,Rákóczi café,cafe,31
6,Резиденція єпископа,Villa Alföld,restaurant,99
7,Реформатська Церква,Комітат Унг,restaurant,43
8,Тарас Шевченко,Їдальня ОДА,restaurant,95
9,Будинок Ужанського комітату (обласний художній...,Комітат Унг,restaurant,70


закладів у шарі: 256 | медіана відстані 70 м, максимум 1010 м


---
## 9. Маршрут на карті

Лінія маршруту — це **геометрія ребер графа**, а не відрізки між пам'ятками.

### Завдання 9.1

Побудуйте карту, на якій **обидві** лінії: справжній маршрут по ребрах і
прямі відрізки між точками. Різниця має бути видно оком.

In [31]:
node_xy = nodes.set_index('osmid')[['x', 'y']]

def route_coords(tour, ids):
    coords = []
    for a, b in zip(tour[:-1], tour[1:]):
        try:
            path = nx.shortest_path(G, ids[a], ids[b], weight='length')
        except nx.NetworkXNoPath:
            path = [ids[a], ids[b]]
        for nd in path:
            coords.append((float(node_xy.loc[nd, 'y']), float(node_xy.loc[nd, 'x'])))
    return coords

real = route_coords(ex10, ids10)                       
straight = [(pts10.geometry.iloc[i].y, pts10.geometry.iloc[i].x) for i in ex10]  

fmap = folium.Map(tiles=BASEMAP)
folium.PolyLine(straight, color='#1f77b4', weight=3, opacity=0.8, dash_array='8',
                tooltip='прямі відрізки').add_to(fmap)
folium.PolyLine(real, color='#d62728', weight=4, opacity=0.9,
                tooltip='маршрут по ребрах графа').add_to(fmap)
for step, i in enumerate(ex10[:-1]):
    g = pts10.geometry.iloc[i]
    folium.Marker((g.y, g.x), tooltip='%d. %s' % (step, pts10['name'].iloc[i]),
                  icon=folium.DivIcon(html='<div style="background:#333;color:#fff;'
                  'border-radius:50%%;width:22px;height:22px;line-height:22px;'
                  'text-align:center;font-weight:bold">%s</div>' % ('С' if step == 0 else step),
                  icon_size=(22, 22), icon_anchor=(11, 11))).add_to(fmap)
fmap.fit_bounds([(min(y for y, _ in real), min(x for _, x in real)),
                 (max(y for y, _ in real), max(x for _, x in real))], padding=(30, 30))
fmap.save('lr5_маршрут.html')

L_straight = sum(((np.hypot(*(np.array(pts10.to_crs(32635).geometry.iloc[a].coords[0])
                              - np.array(pts10.to_crs(32635).geometry.iloc[b].coords[0]))))
                  for a, b in zip(ex10[:-1], ex10[1:])))
print('по графу %.2f км | прямими %.2f км | у %.2f раза довше' % (L_opt / 1000, L_straight / 1000, L_opt / L_straight))
fmap


по графу 13.50 км | прямими 9.77 км | у 1.38 раза довше


---
## 10. Результати для лабораторної роботи № 8

Збережіть файл, який ЛР8 прочитає і спробує побити.

In [32]:
result = {
    'студент': STUDENT_NAME,
    'номер': LIST_NUMBER,
    'місто': CITY,
    'мала_задача': {
        'точок': len(ids10),
        'оптимум_м': round(L_opt, 1),
        'оптимальний_порядок': [str(pts10['name'].iloc[i]) for i in ex10],
        'найближчий_сусід_м': round(tour_length(D10, nn10), 1),
        'двоопт_м': round(tour_length(D10, op10), 1),
    },
    'велика_задача': {
        'точок': len(ids30),
        'найближчий_сусід_м': round(L_nn30, 1),
        'двоопт_м': round(L_op30, 1),
        'двоопт_с': round(t_op30, 4),
        'точки': [{'name': str(pts30['name'].iloc[i]),
                   'lat': float(pts30.geometry.y.iloc[i]),
                   'lon': float(pts30.geometry.x.iloc[i])} for i in range(len(pts30))],
        'матриця_віддалей_м': [[None if not np.isfinite(v) else round(float(v), 1)
                                for v in row] for row in D30],
    },
}
with open('lr5_результати.json', 'w', encoding='utf-8') as f:
    json.dump(result, f, ensure_ascii=False, indent=1)
print('збережено lr5_результати.json (%.1f КБ)' % (os.path.getsize('lr5_результати.json') / 1024))

збережено lr5_результати.json (16.1 КБ)


---
## 11. Звіт

Відповіді пишіть просто в наступній комірці, числами зі своїх обчислень.

1. **Скільки об'єктів ви відкинули** на кроці 2 і за яким правилом? Чи погодились
   би ви віддати цей фільтр замовнику без пояснень?
2. **Які сигнали ввійшли у ваш рейтинг і з якими вагами?** Чому саме так?
3. **Скільки об'єктів збіглося** у ваших двох рейтингах із десяти? Що це
   означає для слова «найкращі»?
4. **Який у вас коефіцієнт обходу** і чи типовий він? Якщо ні — що в місті
   його пояснює?
5. **На скільки відсотків** найближчий сусід гірший за точний розв'язок?
   Чи здивувало вас це число?
6. **Чому точний розв'язок на 30 точках недоступний?** Назвіть конкретну
   величину, яка цьому заважає.
7. **Що у ваших даних виявилось неправильним** — дублікати, пропущені назви,
   об'єкти не на місці? Як ви з цим вчинили?

### Декларація про використання штучного інтелекту

Зазначте, чи користувались ШІ-помічником, у яких саме місцях і як перевіряли
отриманий код. Використання дозволене; **неперевірене** використання — ні.

In [34]:
REPORT = '''
1. Відкинуто 36 з 206 об’єктів шару heritage. Шар уже завантажено за тегами historic або
   tourism, тож за тегами пройшли всі 206, а правило відсіву одне: немає name. Пам’ятку без
   назви туристові не покажеш і не перевіриш, що це, тому правило розумне. Але без пояснень
   я б фільтр не віддала, бо він помиляється в обидва боки. Відкидає справжню історію: 7 фрагментів міських мурів і 2 брами
   не мають назви тому, що в OSM названо цілу споруду, а не кожен шматок. І пропускає шум:
   серед 170 кандидатів 82 artwork і 39 memorial, тобто меморіальні дошки, придорожні хрести
   й «Рамочна майстерня, арт галерея». Фільтр визначає, хто може претендувати, а не хто вартий.
   Відсіювати мають уже ваги рейтингу, і замовник повинен про це знати.
2. score = 0.35·wiki + 0.25·rank(n_lang) + 0.20·museum + 0.15·historic + 0.05·rank(площа).
   Сума ваг 1, тож score від 0 до 1. Ваги спадають за тим, наскільки сигналу можна вірити.
   • wiki, 0.35 — єдиний сигнал ззовні OSM: хтось незалежний визнав об’єкт вартим опису.
     0.5 за елемент Wikidata і ще 0.5 за статтю у Вікіпедії, бо елемент Wikidata буває
     створений ботом під час масового імпорту: в Ужгороді 11 із 36 таких елементів без статті.
   • переклади назви, 0.25 — попит іноземців. Менше, ніж wiki, бо переклади часто беруть
     із Wikidata, і з більшою вагою та сама «слава» рахувалась би двічі. name:uk не враховую:
     у всіх 165 об’єктах, де він є, він дослівно дорівнює name.
   • музей / галерея, 0.20 — для туристичного застосунку важливо, що об’єкт призначений для
     відвідування. Але цей прапорець має і кожна хата скансену, і «Рамочна майстерня, арт галерея»,
     тому про цінність сам по собі він не говорить і стоїть нижче за зовнішні сигнали.
   • historic, 0.15 — відсікає сучасне, але заповнений у 74 зі 170 кандидатів, разом із
     меморіальними дошками й придорожніми хрестами, тож розрізняє слабко.
   • площа, 0.05 — найслабший сигнал: 133 зі 170 кандидатів — точки з площею 0, тож після
     rank(pct=True) площа лише відрізняє будівлю від точки й розводить нічиї.
3. Без злиття дублікатів (завдання 3.2) збіглось 7 з 10, після злиття (крок 3.3) — 8 з 10.
   Сирий рейтинг за площею тричі рахує одне місце: краєзнавчий музей, Ужгородський замок і
   руїни церкви св. Юрія — це одна зупинка, а Будинок правління Ужанського комітату — та сама
   будівля, що й обласний художній музей. Після злиття лише за площею до десятки потрапили
   Липова алея і Будинок Іштвана Дєндеші, лише за моїм критерієм — Горянська ротонда (239 м²)
   і пам’ятник Шевченку (точка, площа її не бачить узагалі). Площа відсікає
   все точкове, а великий парк чи алея в неї піднімаються лише через розмір, не через відомість.
   «Найкращі» в даних не записано: список задає той, хто обирає критерій. В Ужгороді великі
   споруди центру водночас і найвідоміші, тому списки близькі. Розходяться вони на краях, де
   в OSM є точкові об’єкти й великі несуттєві полігони, і саме там видно, що вимірює кожен критерій.
4. Коефіцієнт обходу (медіана відношення «графом / прямою») — 1.34 на 10 точках і 1.32 на 30.
   Це типово для пішохідної мережі (1.2–1.6). Асиметрія 0 м: у пішохода немає одностороннього
   руху. Медіана ховає розкид (930 пар на 30 точках): для пар ближче 300 м вона 1.80, для
   300 м – 1 км — 1.36, для пар далі 1 км — 1.24; 19 % пар мають коефіцієнт понад 1.6. На коротких відстанях будь-який обхід кварталу, огорожі
   чи річки великий порівняно з прямою. Тому маршрут із 10 точок по графу
   має 13.50 км проти 9.77 км прямими, тобто в 1.38 раза довше, більше за медіану: у ньому
   багато коротких переходів у щільному центрі, де пряма лінія бреше найбільше.
5. Найближчий сусід гірший за точний розв’язок на 6.93 % (14 431 м проти 13 496 м, різниця
   близько 0.9 км). 2-opt, що стартує з цього маршруту, гірший лише на 0.53 % (13 568 м): він
   прибрав майже весь надлишок. На 30 точках 2-opt коротший за найближчого сусіда на 7.0 %
   (19 116 м проти 20 561 м) і працює 0.03 с. Число не здивувало: жадібний вибір «найближчої»
   точки на кожному кроці не бачить наслідків, і в кінці маршрут змушений довго повертатись до
   вокзалу залишками, які розкидані по місту. Здивувало інше: наскільки мало бракує 2-opt до
   оптимуму на 10 точках, тож на такому розмірі він майже не поступається точному методу.
   Для 30 точок оптимум невідомий, тому 0.53 % не можна переносити на велику задачу.
6. Заважає розмір таблиці Held–Karp: 2^(n−1)·(n−1) клітин. Для n = 11 це 10 240 клітин, для
   n = 21 — 20.97 млн (0.17 ГБ), для n = 26 — 838.9 млн (6.7 ГБ), а для n = 31 (вокзал і 30
   пам’яток) — 32.2 млрд клітин, тобто близько 258 ГБ при 8 байтах на клітину. Це не
   вміщується в оперативну пам’ять звичайного комп’ютера (8–32 ГБ). Час теж росте за
   O(2^n·n²), але первинна перешкода — саме пам’ять.
7. Неправильного в даних знайшлося п’ять видів, з кожним вчинила по-своєму.
   • Дублікати: одне місце описане кількома об’єктами. У top_score (30 найкращих) знайдено
     9 пар ближче 60 м або вкладених одна в одну. Справжні дублікати — 5 пар, що дають 3 зупинки, зливаю:
     краєзнавчий музей, Ужгородський замок і руїни церкви св. Юрія — одна зупинка
     «Ужгородський замок (краєзнавчий музей)»; обласний художній музей — точка всередині
     будівлі Ужанського комітату; танки Т-70 і Т-34-85 (13 м) — один меморіальний майданчик.
     Решта 4 пари — різні споруди поруч, лишаю окремо: собор і резиденція єпископа (33 м),
     Реформатська церква і художній музей (46 м) та будинок комітату (50 м), пам’ятник
     Августину Волошину всередині Липової алеї. Зливаю, а не видаляю: назви поглинених
     об’єктів лишаються в колонці «злито». На звільнені місця прийшли наступні кандидати,
     нових пар серед них не з’явилось.
   • Пропущені назви: 36 з 206 об’єктів без name — відкинула (п. 1). Свідомо втрачаю
     справжню історію: 7 фрагментів міських мурів і 2 брами.
   • Об’єкти не там, де очікуєш: вокзал у OSM — це дві речі, точка railway=station «Ужгород»
     і будівля «Вокзал Ужгород» за 53 м від неї; старт беру з точки станції. Танк ІС-3 стоїть
     за 118.5 м від найближчого пішохідного вузла (решта точок — до 75 м): доріжок до нього
     в OSM немає, тому віддалі до нього занижені на ці ~120 м в кожен бік. Лишила, бо на
     довжині маршруту на 30 точках (19 км) це близько 1 %. Злипань при прив’язці немає:
     11 і 31 точка займають 11 і 31 різний вузол.

Додатково, з кроків 1 і 7:
• Знімок проти живого запиту: читання знімка швидше приблизно в 900 разів (0.40 с проти 360.9 с; точне число
  трохи міняється від запуску до запуску),
  при повторному запуску мережею не передано 4.20 МБ.
• H3 (рівень 9, ребро ≈ 201 м): 170 кандидатів лежать у 39 комірках, половина з них (85) —
  лише в 7 найщільніших (18 % зайнятих), у найщільнішій 23 пам’ятки. Отже маршрут —
  щільне ядро в старому центрі, де відстані короткі й обхід кварталів найбільший, плюс
  кілька далеких «викидів»: Горянська ротонда стоїть окремо, до найближчого кафе 1010 м.

Декларація про використання ШІ:
Використано Claude Code для написання коду в завданнях, перевірено код, прибрано зайві обчислення і перевірено результати
'''
print(REPORT)


1. Відкинуто 36 з 206 об’єктів шару heritage. Шар уже завантажено за тегами historic або
   tourism, тож за тегами пройшли всі 206, а правило відсіву одне: немає name. Пам’ятку без
   назви туристові не покажеш і не перевіриш, що це, тому правило розумне. Але без пояснень
   я б фільтр не віддала, бо він помиляється в обидва боки. Відкидає справжню історію: 7 фрагментів міських мурів і 2 брами
   не мають назви тому, що в OSM названо цілу споруду, а не кожен шматок. І пропускає шум:
   серед 170 кандидатів 82 artwork і 39 memorial, тобто меморіальні дошки, придорожні хрести
   й «Рамочна майстерня, арт галерея». Фільтр визначає, хто може претендувати, а не хто вартий.
   Відсіювати мають уже ваги рейтингу, і замовник повинен про це знати.
2. score = 0.35·wiki + 0.25·rank(n_lang) + 0.20·museum + 0.15·historic + 0.05·rank(площа).
   Сума ваг 1, тож score від 0 до 1. Ваги спадають за тим, наскільки сигналу можна вірити.
   • wiki, 0.35 — єдиний сигнал ззовні OSM: хтось незалежний визнав 